# 01 · Denoising: band-pass filters vs. DeepDenoiser

Volcanic and geothermal sites are noisy: wind on the caldera rim, traffic, villages, and at
Aluto the geothermal wells and pipelines. Before picking, we ask:

* Where in frequency is the earthquake signal, and where is the noise?
* Does a **neural denoiser** recover more signal than a carefully chosen band-pass filter?
* And, most importantly, **does denoising change what a picker finds?**

**DeepDenoiser** ([Zhu, Mousavi & Beroza, 2019](https://doi.org/10.1109/TGRS.2019.2926772))
works in the short-time Fourier domain. For every time-frequency bin it predicts a mask that
splits the recording into "signal" and "noise", then inverts the masked spectrogram back to a
waveform. Each component is processed on its own. SeisBench ships two sets of weights:
`original` (trained on Northern California data) and `urban` (trained with urban noise from
Long Beach).

We work with the demonstration station only. Picking on the whole network comes in
notebook 02.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seisbench
import seisbench.models as sbm
import torch
from obspy import UTCDateTime
from scipy.signal import welch

import alutoml as aml
from alutoml import data, picking, plotting

plotting.use_style()
cfg = aml.load_config()
torch.set_num_threads(max(1, torch.get_num_threads()))
print(f"SeisBench {seisbench.__version__}, torch {torch.__version__}, "
      f"station {cfg.network}.{cfg.demo_station}, day {cfg.day.date}")

## 1 · Find events to look at

A classic recursive STA/LTA trigger (0.5 s / 10 s, 2–20 Hz) scans the whole day on the
vertical component. We use its strongest triggers only to choose time windows for the
figures. It is our "pre-ML" baseline detector.

In [ ]:
st_day = data.load_waveforms(cfg.sds, cfg.network, cfg.demo_station, cfg.t0, cfg.t1)
z_day = st_day.select(component="Z")          # continuous segments (split at gaps)
triggers = picking.sta_lta_triggers(z_day, sta=0.5, lta=10.0, on=5.0, off=1.5)
print(f"{len(triggers)} STA/LTA triggers on {z_day[0].id} ({len(z_day)} continuous segment(s))")
triggers.head(10)

In [ ]:
best = UTCDateTime(triggers["onset"].iloc[0].timestamp())
w0, w1 = best - 20, best + 40           # one-minute window around the strongest trigger
h0 = UTCDateTime(best.datetime.replace(minute=0, second=0, microsecond=0))
h1 = h0 + 3600                           # the clock hour that contains it
print(f"Strongest trigger: {best}  (window {w0.time} – {w1.time})")

st_hour = data.load_waveforms(cfg.sds, cfg.network, cfg.demo_station, h0 - 60, h1 + 60)
st_hour.detrend("demean")
raw = st_hour.slice(w0, w1)
fig = plotting.waveform_spectrogram(raw, title=f"Raw: {cfg.network}.{cfg.demo_station}")

## 2 · Let the data choose the band

Rather than reaching for the usual "2–20 Hz", we compare the power spectral density (Welch)
of a pre-event noise window with that of the event window. The band where signal stands
above noise is the natural pass-band.

In [ ]:
fs = raw[0].stats.sampling_rate
fig, ax = plt.subplots(figsize=(7, 4))
snr_curves = []
for tr in raw:
    noise = st_hour.select(id=tr.id).slice(best - 18, best - 2)[0].data
    signal = st_hour.select(id=tr.id).slice(best, best + 16)[0].data
    f, pn = welch(noise - noise.mean(), fs=fs, nperseg=256)
    _, ps = welch(signal - signal.mean(), fs=fs, nperseg=256)
    ax.loglog(f, ps, lw=1.4, label=f"{tr.stats.channel} event")
    ax.loglog(f, pn, lw=1.0, ls="--", label=f"{tr.stats.channel} noise")
    snr_curves.append(ps / pn)
snr_f = np.median(snr_curves, axis=0)
ax.set(xlabel="Frequency (Hz)", ylabel="PSD (counts²/Hz)", xlim=(0.5, 50),
       title="Event vs. pre-event noise")
ax.legend(fontsize=7, ncols=3)

good = f[(snr_f > 3) & (f > 0.5)]
fmin, fmax = (float(good.min()), float(min(good.max(), 45))) if good.size else (2.0, 20.0)
print(f"Spectral SNR > 3 between {fmin:.1f} and {fmax:.1f} Hz")

In [ ]:
bandpassed = st_hour.copy().taper(0.01).filter(
    "bandpass", freqmin=max(fmin, 1.0), freqmax=fmax, corners=4, zerophase=True)
fig = plotting.waveform_spectrogram(bandpassed.slice(w0, w1),
                                    title=f"Band-pass {max(fmin, 1.0):.1f}–{fmax:.1f} Hz")

## 3 · DeepDenoiser

`annotate` slides the network over the stream and returns one denoised trace per component.
Output channels are renamed to `DeepDenoiser_HHZ` and so on. We strip that prefix again so the
result can be given to a picker.

In [ ]:
print(sbm.DeepDenoiser.list_pretrained())

In [ ]:
denoised = {}
for weights in ("original", "urban"):
    model = sbm.DeepDenoiser.from_pretrained(weights)
    denoised[weights] = picking.strip_model_prefix(model.annotate(st_hour))
    print(weights, denoised[weights])

In [ ]:
fig = plotting.waveform_spectrogram(denoised["original"].slice(w0, w1),
                                    title="DeepDenoiser (original weights)")

In [ ]:
versions = {
    "raw": st_hour,
    "band-pass": bandpassed,
    "DD-original": denoised["original"],
    "DD-urban": denoised["urban"],
}
fig, axes = plt.subplots(len(versions), 1, figsize=(10, 7), sharex=True)
for ax, (name, st) in zip(axes, versions.items()):
    tr = st.select(component="Z").slice(w0, w1)[0]
    ax.plot(tr.times() - 20, tr.data / np.abs(tr.data).max(), lw=0.5, color="k")
    ax.set_ylabel(name, rotation=0, ha="right", va="center")
    ax.set_yticks([])
axes[-1].set_xlabel("Seconds relative to STA/LTA onset")
axes[0].set_title("Vertical component, each normalised to its maximum");

### Quantifying the gain

For the ten strongest triggers of the hour-long window, compute the RMS signal-to-noise
ratio (5 s after the onset vs. 5 s before) on the vertical component for each version.

In [ ]:
in_hour = triggers[(triggers["onset"] >= pd.Timestamp(h0.datetime, tz="UTC")) &
                   (triggers["onset"] < pd.Timestamp(h1.datetime, tz="UTC"))].head(10)
rows = []
for _, trig in in_hour.iterrows():
    onset = UTCDateTime(trig["onset"].timestamp())
    row = {"onset": trig["onset"].strftime("%H:%M:%S")}
    for name, st in versions.items():
        row[name] = picking.snr_db(st.select(component="Z")[0], onset)
    rows.append(row)
snr = pd.DataFrame(rows).set_index("onset")
snr.round(1)

In [ ]:
gain = snr.drop(columns="raw").sub(snr["raw"], axis=0)
ax = gain.plot.box(figsize=(6, 3.5), grid=True)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("SNR gain over raw (dB)");

**Be careful when reading SNR gains.** The denoiser is trained to suppress whatever does not
look like an earthquake, so the "noise" window is pushed towards zero by design. A high SNR
is not proof that the waveform is faithful. Look for distorted onsets, changed polarities
and ringing before you trust a denoised record for amplitudes or first motions.

## 4 · Does denoising change the picks?

Now the question that matters for a catalogue. We run the same generic picker
(PhaseNet with `instance` weights, trained on Italian seismicity) on the four versions of the
hour and compare.

In [ ]:
picker = picking.ModelSpec("PhaseNet", "instance").load()
pick_tables = {name: picking.run_picker(picker, st, threshold=0.2) for name, st in versions.items()}
summary = pd.DataFrame({
    name: {"P": int((df.phase == "P").sum()), "S": int((df.phase == "S").sum()),
           "median prob.": df["probability"].median(), "runtime (s)": df.attrs["runtime_s"]}
    for name, df in pick_tables.items()}).T
summary

In [ ]:
fig = plotting.probability_histograms(pick_tables)

In [ ]:
mat = picking.agreement_matrix(pick_tables, phase="P", tolerance=0.2)
fig = plotting.matrix_heatmap(mat, title="P picks of row also found in column (±0.2 s)")

In [ ]:
annotations = {name: picker.annotate(st.slice(w0 - 30, w1 + 30)) for name, st in versions.items()}
fig = plotting.multi_model_curves(raw, annotations, title="PhaseNet-instance on each version")

## Take-aways

* A data-driven band-pass is a strong, transparent baseline.
* DeepDenoiser can raise SNR considerably. Whether that yields *more* or *better* picks depends
  on the picker. Modern pickers learn their own noise model, so the extra step can add little,
  and denoising can introduce artefacts the picker was never trained on.
* For the rest of the workshop we pick on **raw** data, as is standard for SeisBench pickers.

## Exercises

1. Repeat section 4 with `ModelSpec("PhaseNet", "volpick")`. Does the volcanic-trained model
   benefit more or less from denoising?
2. Pick a station on the caldera rim (e.g. A03E) and one near the geothermal plant (A09E).
   Where does DeepDenoiser help most?
3. Plot the denoised and raw horizontal components around an S arrival. Is the S onset
   preserved?

**Next:** 02 · Phase picking (`02_phase_picking.ipynb`)